# Glucast — Pre-entrenamiento con OhioT1DM + Fine-tuning con datos propios
## TFM — Maestría en Inteligencia Artificial — UNIR
**Autor:** Pablo Daniel Figueroa

Este notebook implementa la estrategia de transfer learning:
1. **Pre-entrenamiento** con OhioT1DM (12 pacientes, ~8 semanas c/u)
2. **Fine-tuning** con datos propios del autor (CGM + Garmin)


## 1. Instalación

In [ ]:
!pip install pandas numpy matplotlib scikit-learn tensorflow keras seaborn -q
print('✅ Listo')

## 2. Importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import glob, pickle, json, warnings
warnings.filterwarnings('ignore')
print('✅ Importaciones OK')

## 3. Subir el ZIP de OhioT1DM
Subí el archivo `OhioT1DM.zip` (con contraseña).

In [ ]:
from google.colab import files
uploaded = files.upload()

## 4. Descomprimir OhioT1DM

In [ ]:
import zipfile, os
# La contraseña del dataset OhioT1DM se obtiene al firmar el acuerdo de uso (DUA)
# con sus autores. No se incluye en el repositorio.
import getpass
PASSWORD = getpass.getpass('Contraseña del dataset OhioT1DM: ').encode()
with zipfile.ZipFile('OhioT1DM.zip', 'r') as z:
    z.extractall('/content/OhioT1DM', pwd=PASSWORD)
print('✅ Descomprimido')
for f in sorted(glob.glob('/content/OhioT1DM/**/*.xml', recursive=True)):
    print(' ', f.split('OhioT1DM/')[-1])

## 5. Parser de archivos XML de OhioT1DM

In [ ]:
def parsear_ohio_xml(filepath):
    tree = ET.parse(filepath)
    root = tree.getroot()
    patient_id = root.get('id')

    # Glucosa CGM (5 min)
    gluc_rows = []
    for ev in root.find('glucose_level').findall('event'):
        gluc_rows.append({'timestamp': pd.to_datetime(ev.get('ts'), format='%d-%m-%Y %H:%M:%S'),
                          'glucosa': float(ev.get('value'))})

    # Bolos de insulina
    bolo_rows = []
    bolus_el = root.find('bolus')
    if bolus_el is not None:
        for ev in bolus_el.findall('event'):
            bolo_rows.append({'timestamp': pd.to_datetime(ev.get('ts_begin'), format='%d-%m-%Y %H:%M:%S'),
                              'bolo_u': float(ev.get('dose', 0))})

    # Comidas (carbohidratos)
    meal_rows = []
    meal_el = root.find('meal')
    if meal_el is not None:
        for ev in meal_el.findall('event'):
            carbs = ev.get('carbs')
            if carbs:
                meal_rows.append({'timestamp': pd.to_datetime(ev.get('ts'), format='%d-%m-%Y %H:%M:%S'),
                                  'carbs': float(carbs)})

    # FC (Basis Band)
    hr_rows = []
    hr_el = root.find('basis_heart_rate')
    if hr_el is not None:
        for ev in hr_el.findall('event'):
            hr_rows.append({'timestamp': pd.to_datetime(ev.get('ts'), format='%d-%m-%Y %H:%M:%S'),
                            'fc': float(ev.get('value'))})

    # Pasos
    steps_rows = []
    steps_el = root.find('basis_steps')
    if steps_el is not None:
        for ev in steps_el.findall('event'):
            steps_rows.append({'timestamp': pd.to_datetime(ev.get('ts'), format='%d-%m-%Y %H:%M:%S'),
                               'pasos': float(ev.get('value'))})

    # Construir DataFrame base (5 min)
    df_gluc = pd.DataFrame(gluc_rows).set_index('timestamp').sort_index()
    df_base = df_gluc.resample('5min').mean().interpolate(method='linear', limit=3)

    # Agregar bolos
    if bolo_rows:
        df_bolos = pd.DataFrame(bolo_rows).set_index('timestamp').sort_index()
        df_bolos_5m = df_bolos.resample('5min').sum()
        df_base = df_base.join(df_bolos_5m, how='left')
        df_base['bolo_u'] = df_base['bolo_u'].fillna(0)
    else:
        df_base['bolo_u'] = 0

    # Agregar carbs
    if meal_rows:
        df_meals = pd.DataFrame(meal_rows).set_index('timestamp').sort_index()
        df_meals_5m = df_meals.resample('5min').sum()
        df_base = df_base.join(df_meals_5m, how='left')
        df_base['carbs'] = df_base['carbs'].fillna(0)
    else:
        df_base['carbs'] = 0

    # FC a resolución 5min
    if hr_rows:
        df_hr = pd.DataFrame(hr_rows).set_index('timestamp').sort_index()
        df_hr_5m = df_hr.resample('5min').mean().interpolate(method='linear', limit=12)
        df_base = df_base.join(df_hr_5m, how='left')
        df_base['fc'] = df_base['fc'].fillna(df_base['fc'].mean())
    else:
        df_base['fc'] = 75

    # Pasos a resolución 5min
    if steps_rows:
        df_steps = pd.DataFrame(steps_rows).set_index('timestamp').sort_index()
        df_steps_5m = df_steps.resample('5min').sum()
        df_base = df_base.join(df_steps_5m, how='left')
        df_base['pasos'] = df_base['pasos'].fillna(0)
    else:
        df_base['pasos'] = 0

    # IOB estimado (DIA=2h=24 periodos)
    DIA = 24
    pesos = np.array([max(0, 1 - i/DIA) for i in range(DIA)])
    df_base['iob'] = df_base['bolo_u'].rolling(DIA, min_periods=1).apply(
        lambda x: np.sum(x[::-1] * pesos[:len(x)]), raw=True)

    # Tendencia
    df_base['tendencia'] = df_base['glucosa'].diff().fillna(0)

    # Hora ciclica
    hora = df_base.index.hour + df_base.index.minute / 60
    df_base['hora_sin'] = np.sin(2 * np.pi * hora / 24)
    df_base['hora_cos'] = np.cos(2 * np.pi * hora / 24)

    # Dia semana ciclico
    dow = df_base.index.dayofweek
    df_base['dow_sin'] = np.sin(2 * np.pi * dow / 7)
    df_base['dow_cos'] = np.cos(2 * np.pi * dow / 7)

    # Fenomeno del alba
    df_base['alba'] = ((df_base.index.hour >= 5) & (df_base.index.hour < 8)).astype(int)

    df_base['patient_id'] = patient_id
    df_base = df_base.dropna(subset=['glucosa'])
    return df_base

print('✅ Parser definido')

## 6. Cargar todos los pacientes de OhioT1DM

In [ ]:
FEATURES = ['glucosa', 'bolo_u', 'iob', 'tendencia', 'carbs',
            'hora_sin', 'hora_cos', 'dow_sin', 'dow_cos', 'alba', 'fc', 'pasos']
TARGET = 'glucosa'
VENTANA = 24
HORIZONTE = 6

train_files = sorted(glob.glob('/content/OhioT1DM/**/train/*.xml', recursive=True))
test_files = sorted(glob.glob('/content/OhioT1DM/**/test/*.xml', recursive=True))

print(f'Archivos train: {len(train_files)}')
print(f'Archivos test:  {len(test_files)}')

def crear_secuencias(df, features, target, ventana, horizonte):
    data = df[features].dropna().values
    target_idx = features.index(target)
    X, y = [], []
    for i in range(ventana, len(data) - horizonte):
        X.append(data[i-ventana:i])
        y.append(data[i+horizonte-1, target_idx])
    return np.array(X), np.array(y)

def cargar_pacientes(archivos, features, ventana, horizonte, scaler=None):
    X_all, y_all = [], []
    scalers = []
    for f in archivos:
        pid = f.split('/')[-1].split('-')[0]
        try:
            df = parsear_ohio_xml(f)
            df_feat = df[features].fillna(method='ffill').fillna(0)
            sc = MinMaxScaler()
            data_sc = sc.fit_transform(df_feat)
            scalers.append((pid, sc))
            target_idx = features.index(TARGET)
            for i in range(ventana, len(data_sc) - horizonte):
                X_all.append(data_sc[i-ventana:i])
                y_all.append(data_sc[i+horizonte-1, target_idx])
            print(f'  Paciente {pid}: {len(df)} filas')
        except Exception as e:
            print(f'  Error {pid}: {e}')
    return np.array(X_all), np.array(y_all), scalers

print('\nCargando datos de entrenamiento OhioT1DM...')
X_ohio, y_ohio, scalers_ohio = cargar_pacientes(train_files, FEATURES, VENTANA, HORIZONTE)
print(f'\nTotal secuencias OhioT1DM train: {len(X_ohio)}')
print(f'Shape X: {X_ohio.shape}')

## 7. Pre-entrenamiento con OhioT1DM (12 pacientes)

In [ ]:
def construir_modelo(input_shape):
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        BatchNormalization(), Dropout(0.2),
        LSTM(32, return_sequences=False),
        BatchNormalization(), Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

modelo_pretrain = construir_modelo((VENTANA, len(FEATURES)))
modelo_pretrain.summary()

In [ ]:
split = int(len(X_ohio) * 0.85)
X_tr, X_val = X_ohio[:split], X_ohio[split:]
y_tr, y_val = y_ohio[:split], y_ohio[split:]

callbacks_pretrain = [
    EarlyStopping(patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(patience=5, factor=0.5, min_lr=1e-6, verbose=1),
    ModelCheckpoint('glucast_pretrain_ohio.keras', save_best_only=True, verbose=0)
]

hist_pretrain = modelo_pretrain.fit(
    X_tr, y_tr,
    epochs=80,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=callbacks_pretrain,
    verbose=1
)
print('\n✅ Pre-entrenamiento OhioT1DM completado')

## 8. Curvas de aprendizaje — Pre-entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(hist_pretrain.history['loss'], label='Train')
axes[0].plot(hist_pretrain.history['val_loss'], label='Val')
axes[0].set_title('Loss MSE — Pre-entrenamiento OhioT1DM')
axes[0].legend()
axes[1].plot(hist_pretrain.history['mae'], label='Train')
axes[1].plot(hist_pretrain.history['val_mae'], label='Val')
axes[1].set_title('MAE — Pre-entrenamiento OhioT1DM')
axes[1].legend()
plt.tight_layout()
plt.savefig('pretrain_curvas.png', dpi=150)
plt.show()

## 9. Subir datos propios (CareLink + Garmin) para fine-tuning

In [ ]:
print('Subí los archivos:')
print('  1. CSV de CareLink (Pablo Figueroa...csv)')
print('  2. garmin_datos_consolidados.csv')
uploaded2 = files.upload()

## 10. Preparar datos propios para fine-tuning

In [ ]:
import csv
from datetime import datetime

def parsear_carelink(filename):
    glucosa_rows, bolo_rows = [], []
    headers = None
    date_col = time_col = sgv_col = bolus_col = None
    with open(filename, encoding='utf-8-sig') as f:
        reader = csv.reader(f)
        rows = list(reader)
    for row in rows:
        if not row: continue
        if 'Date' in row and 'Time' in row and 'Sensor Glucose (mg/dL)' in row:
            headers = row
            date_col = headers.index('Date')
            time_col = headers.index('Time')
            sgv_col = headers.index('Sensor Glucose (mg/dL)')
            bolus_col = headers.index('Bolus Volume Delivered (U)') if 'Bolus Volume Delivered (U)' in headers else None
            continue
        if headers is None: continue
        try:
            dt = datetime.strptime(f"{row[date_col].strip()} {row[time_col].strip()}", '%Y/%m/%d %H:%M:%S')
        except: continue
        if row[sgv_col].strip().isdigit():
            glucosa_rows.append({'timestamp': dt, 'glucosa': int(row[sgv_col])})
        if bolus_col and bolus_col < len(row) and row[bolus_col].strip():
            try:
                u = float(row[bolus_col])
                if u > 0: bolo_rows.append({'timestamp': dt, 'bolo_u': u})
            except: pass
    return pd.DataFrame(glucosa_rows).set_index('timestamp').sort_index(), \
           pd.DataFrame(bolo_rows).set_index('timestamp').sort_index()

carelink_file = [f for f in uploaded2.keys() if 'Figueroa' in f or 'Pablo' in f][0]
garmin_file   = [f for f in uploaded2.keys() if 'garmin' in f.lower()][0]
print(f'CareLink: {carelink_file}')
print(f'Garmin:   {garmin_file}')

In [ ]:
df_gluc, df_bolos = parsear_carelink(carelink_file)
df_garmin = pd.read_csv(garmin_file)
df_garmin['fecha'] = pd.to_datetime(df_garmin['fecha'])
df_garmin = df_garmin.set_index('fecha').sort_index()

# Construir base en 5 min
df_base = df_gluc.resample('5min').mean().interpolate(method='linear', limit=3)
if len(df_bolos):
    df_base = df_base.join(df_bolos.resample('5min').sum(), how='left')
df_base['bolo_u'] = df_base.get('bolo_u', pd.Series(0, index=df_base.index)).fillna(0)
df_base['carbs'] = 0  # no hay carbs en CareLink

# IOB
pesos_iob = np.array([max(0, 1 - i/24) for i in range(24)])
df_base['iob'] = df_base['bolo_u'].rolling(24, min_periods=1).apply(
    lambda x: np.sum(x[::-1] * pesos_iob[:len(x)]), raw=True)
df_base['tendencia'] = df_base['glucosa'].diff().fillna(0)
hora = df_base.index.hour + df_base.index.minute/60
df_base['hora_sin'] = np.sin(2*np.pi*hora/24)
df_base['hora_cos'] = np.cos(2*np.pi*hora/24)
dow = df_base.index.dayofweek
df_base['dow_sin'] = np.sin(2*np.pi*dow/7)
df_base['dow_cos'] = np.cos(2*np.pi*dow/7)
df_base['alba'] = ((df_base.index.hour>=5)&(df_base.index.hour<8)).astype(int)
df_base['fecha'] = df_base.index.normalize()

# Garmin
cols_g = ['estres_promedio','fc_reposo','sueno_total_horas','pasos_totales']
cols_g = [c for c in cols_g if c in df_garmin.columns]
mapping = {'estres_promedio':'estres','fc_reposo':'fc','sueno_total_horas':'sueno','pasos_totales':'pasos'}
df_g2 = df_garmin[cols_g].rename(columns=mapping).reset_index()
df_g2.columns = ['fecha'] + list(df_g2.columns[1:])
df_base = df_base.reset_index().merge(df_g2, on='fecha', how='left').set_index('timestamp')
df_base = df_base.drop(columns=['fecha'])

# Renombrar columnas Garmin para que coincidan con Ohio features
if 'fc' not in df_base.columns and 'fc_reposo' in df_base.columns:
    df_base['fc'] = df_base['fc_reposo']
if 'pasos' not in df_base.columns and 'pasos_totales' in df_base.columns:
    df_base['pasos'] = df_base['pasos_totales']
if 'fc' not in df_base.columns:
    df_base['fc'] = 72
if 'pasos' not in df_base.columns:
    df_base['pasos'] = 0

df_base['fc'] = df_base['fc'].fillna(df_base['fc'].mean())
df_base['pasos'] = df_base['pasos'].fillna(0)
df_base = df_base.dropna(subset=['glucosa'])
print(f'Dataset propio: {len(df_base)} filas, {len(df_base.columns)} columnas')
print(df_base[FEATURES].isnull().sum())

## 11. Fine-tuning con datos propios

In [ ]:
# Escalar datos propios
df_model = df_base[FEATURES].fillna(0)
scaler_finetune = MinMaxScaler()
data_sc = scaler_finetune.fit_transform(df_model)
target_idx = FEATURES.index(TARGET)

X_ft, y_ft = [], []
for i in range(VENTANA, len(data_sc)-HORIZONTE):
    X_ft.append(data_sc[i-VENTANA:i])
    y_ft.append(data_sc[i+HORIZONTE-1, target_idx])
X_ft, y_ft = np.array(X_ft), np.array(y_ft)
split = int(len(X_ft)*0.8)
X_train_ft, X_test_ft = X_ft[:split], X_ft[split:]
y_train_ft, y_test_ft = y_ft[:split], y_ft[split:]
print(f'Fine-tuning — Train: {len(X_train_ft)} | Test: {len(X_test_ft)}')

In [ ]:
# Cargar modelo pre-entrenado y hacer fine-tuning
modelo_ft = load_model('glucast_pretrain_ohio.keras')

# Reducir learning rate para fine-tuning
modelo_ft.compile(optimizer=Adam(learning_rate=0.0001), loss='mse', metrics=['mae'])

callbacks_ft = [
    EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(patience=7, factor=0.5, min_lr=1e-7, verbose=1),
    ModelCheckpoint('glucast_modelo_finetune.keras', save_best_only=True, verbose=0)
]

hist_ft = modelo_ft.fit(
    X_train_ft, y_train_ft,
    epochs=100,
    batch_size=32,
    validation_split=0.15,
    callbacks=callbacks_ft,
    verbose=1
)
print('\n✅ Fine-tuning completado')

## 12. Evaluación comparativa — antes vs después del fine-tuning

In [ ]:
def evaluar(modelo, X_test, y_test, scaler, target_idx, nombre):
    y_pred_sc = modelo.predict(X_test, verbose=0)
    n = len(FEATURES)
    def invertir(vals):
        d = np.zeros((len(vals), n))
        d[:, target_idx] = vals.flatten()
        return scaler.inverse_transform(d)[:, target_idx]
    y_pred = invertir(y_pred_sc)
    y_real = invertir(y_test)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    mae = mean_absolute_error(y_real, y_pred)
    zona_a = np.mean((np.abs(y_real-y_pred)/np.maximum(y_real,1)<0.20)|((y_real<70)&(y_pred<70)))*100
    tir_real = np.mean((y_real>=70)&(y_real<=180))*100
    tir_pred = np.mean((y_pred>=70)&(y_pred<=180))*100
    print(f'\n--- {nombre} ---')
    print(f'RMSE:          {rmse:.2f} mg/dL')
    print(f'MAE:           {mae:.2f} mg/dL')
    print(f'Clarke Zona A: {zona_a:.1f}%')
    print(f'TIR real:      {tir_real:.1f}%')
    print(f'TIR predicho:  {tir_pred:.1f}%')
    return y_real, y_pred, rmse, mae

# Evaluar modelo base (1er entrenamiento)
try:
    modelo_base = load_model('glucast_modelo_lstm.h5')
    y_r0, y_p0, rmse0, mae0 = evaluar(modelo_base, X_test_ft, y_test_ft, scaler_finetune, target_idx, 'Modelo base (solo datos propios)')
except:
    print('(No se encontró glucast_modelo_lstm.h5 — saltando comparación con modelo base)')
    y_r0, y_p0, rmse0, mae0 = None, None, None, None

# Evaluar modelo fine-tuned
y_real, y_pred, rmse_ft, mae_ft = evaluar(modelo_ft, X_test_ft, y_test_ft, scaler_finetune, target_idx, 'Modelo fine-tuned (OhioT1DM + datos propios)')

## 13. Visualización comparativa

In [ ]:
n = min(500, len(y_real))
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(y_real[:n], label='Glucosa real', color='#1a73e8', linewidth=1)
axes[0].plot(y_pred[:n], label=f'Fine-tuned RMSE={rmse_ft:.1f}', color='#e53935', linewidth=1, linestyle='--')
if y_p0 is not None:
    axes[0].plot(y_p0[:n], label=f'Base RMSE={rmse0:.1f}', color='#43a047', linewidth=1, linestyle=':')
axes[0].axhline(70, color='red', alpha=0.3, linestyle=':')
axes[0].axhline(180, color='orange', alpha=0.3, linestyle=':')
axes[0].fill_between(range(n), 70, 180, alpha=0.05, color='green')
axes[0].set_ylabel('Glucosa (mg/dL)')
axes[0].set_title('Predicción a 30 min — Comparativa base vs fine-tuned')
axes[0].legend()

errores_ft = y_pred[:n] - y_real[:n]
axes[1].bar(range(len(errores_ft)), errores_ft,
            color=np.where(np.abs(errores_ft)>20,'#e53935','#43a047'), alpha=0.6, width=1)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].axhline(20, color='orange', linestyle='--', alpha=0.5)
axes[1].axhline(-20, color='orange', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Error (mg/dL)')
axes[1].set_title('Error de predicción (verde < 20 mg/dL)')
plt.tight_layout()
plt.savefig('finetune_predicciones.png', dpi=150)
plt.show()

## 14. Guardar modelo final y descargar

In [ ]:
metadata_ft = {
    'modelo': 'LSTM fine-tuned OhioT1DM + datos propios',
    'features': FEATURES,
    'ventana': VENTANA,
    'horizonte': HORIZONTE,
    'rmse': float(rmse_ft),
    'mae': float(mae_ft),
    'n_pacientes_pretrain': 12,
    'n_seq_finetune_train': len(X_train_ft),
    'n_seq_finetune_test': len(X_test_ft),
}
with open('glucast_finetune_metadata.json', 'w') as f:
    json.dump(metadata_ft, f, indent=2)
with open('glucast_finetune_scaler.pkl', 'wb') as f:
    pickle.dump(scaler_finetune, f)
print('✅ Archivos guardados')
print(json.dumps(metadata_ft, indent=2))

In [ ]:
from google.colab import files
files.download('glucast_modelo_finetune.keras')
files.download('glucast_finetune_scaler.pkl')
files.download('glucast_finetune_metadata.json')
files.download('finetune_predicciones.png')
files.download('pretrain_curvas.png')
print('✅ Descargados')

## 15. Resumen final

**Estrategia implementada:**
- **Pre-entrenamiento** con OhioT1DM (12 pacientes con DMT1, ~8 semanas c/u, datos 2018 y 2020)
- **Fine-tuning** con datos propios del autor (CGM CareLink + Garmin Instinct)

**Ventajas del transfer learning:**
- El modelo aprende patrones glucémicos generales de 12 pacientes distintos
- El fine-tuning adapta el modelo a los patrones específicos del autor
- Se espera mejor generalización y menor RMSE que el modelo entrenado solo con datos propios

**Próximos pasos:**
- Comparar RMSE antes y después del fine-tuning
- Integrar el modelo en la app Glucast
- Reentrenar mensualmente con datos acumulados
